# نوفا الصغير — التدريب على Google Colab (نسخة موازية لدفتر Kaggle)

هذا الدفتر **يفعل نفس التدريب بالضبط** الذي يفعله دفتر Kaggle (`sham_small_training.ipynb`)، لكن على Google Colab بدل Kaggle. السبب: Colab لا ينشر أي سقف أسبوعي أو يومي ثابت للاستخدام (بعكس سقف Kaggle الصريح 30 ساعة/أسبوع) — لكنه لا يزال له حد لطول الجلسة الواحدة (~12 ساعة) وقد يبطئ أولويتك تدريجياً مع استخدام مكثف جداً بلا انقطاع لأيام متتالية (نظام غير معلن من جوجل).

**الفرق الأهم عملياً:** الاستئناف هنا أبسط من Kaggle — لا حاجة لإنشاء أو رفع أي Dataset يدوياً؛ كل تقدّم التدريب يُحفظ مباشرة في **Google Drive الخاص بك**، فتفتح الدفتر في أي وقت تريده وتضغط **Run All** فقط، وهو يكمل من نفسه تلقائياً.

## قبل الضغط على "Run All" — مرة واحدة فقط:

1. **أضف رمز وصول GitHub كـ Colab Secret:**
   - من GitHub: `Settings → Developer settings → Personal access tokens → Generate new token` (صلاحية `repo` للقراءة فقط تكفي).
   - في Colab: اضغط أيقونة **🔑 (المفتاح)** في الشريط الجانبي الأيسر → **Add new secret** → الاسم `GITHUB_TOKEN` والقيمة هي الرمز. **مهم:** فعّل مفتاح "Notebook access" بجانبه حتى يسمح لهذا الدفتر بقراءته.
2. **اختر معالج رسومي (GPU):** من القائمة العلوية **Runtime → Change runtime type → Hardware accelerator → GPU (T4)**.
3. عند تشغيل خلية ربط Drive أدناه لأول مرة، ستظهر لك نافذة لتسجيل الدخول بحساب جوجل والموافقة على الوصول — وافق عليها (هذا يربط Drive الخاص بك فقط، ولا يُشارك مع أي طرف آخر).

بعد هذه الخطوات، اضغط **Runtime → Run all** ودع الدفتر يعمل من أوله لآخره.

### 1) ربط Google Drive (لحفظ التقدّم بشكل دائم)

In [1]:
from google.colab import drive
import os

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ShamSmall"
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
print(f"تم ربط Google Drive بنجاح — كل تقدّم التدريب سيُحفظ بشكل دائم هنا: {DRIVE_ROOT}")
print("لن تحتاج لتنزيل أو رفع أي شيء يدوياً بين الجلسات — فقط أعد فتح هذا الدفتر واضغط Run all.")


Mounted at /content/drive
تم ربط Google Drive بنجاح — كل تقدّم التدريب سيُحفظ بشكل دائم هنا: /content/drive/MyDrive/ShamSmall
لن تحتاج لتنزيل أو رفع أي شيء يدوياً بين الجلسات — فقط أعد فتح هذا الدفتر واضغط Run all.


### 2) سحب الكود الحقيقي من GitHub مباشرة

In [2]:
import os
import sys
import subprocess
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
# هذا الكود لا يزال على فرع (branch) العمل الحالي، وليس على الفرع الرئيسي main بعد —
# إن دُمج لاحقاً إلى main يمكن حذف --branch هذا بأمان.
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/content/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


تم سحب الكود الحقيقي من مستودع GitHub بنجاح.
كود ShamSmall الحقيقي جاهز في: /content/Ttbik/ai-system/colab/sham_small


### 3) تثبيت المكتبات الإضافية غير الموجودة افتراضياً على Colab

In [3]:
try:
    import tokenizers
    print(f"مكتبة tokenizers متوفرة مسبقاً (نسخة {tokenizers.__version__}) — لا حاجة للتثبيت.")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
    print("تم تثبيت مكتبة tokenizers.")


مكتبة tokenizers متوفرة مسبقاً (نسخة 0.23.1) — لا حاجة للتثبيت.


### 4) جمع بيانات نصية عربية حقيقية

هذه الخلية تسحب (streaming) شريحة حقيقية من **ويكيبيديا العربية**. تُجمع من جديد في كل جلسة (بيانات مصدرها الإنترنت، وليست مما نحفظه في Drive) — هذا طبيعي ولا يستغرق وقتاً يُذكر.

In [5]:
from data_acquisition import stream_hf_text_corpus

MAX_DOCUMENTS = 20_000  # ابدأ بعدد معقول للتأكد أن كل شيء يعمل، ثم كبّره في تشغيل لاحق

corpus_dir = "/content/corpus/wikipedia_ar"
corpus_files = stream_hf_text_corpus(
    dataset_name="wikimedia/wikipedia",
    config_name="20231101.ar",
    text_field="text",
    output_dir=corpus_dir,
    max_documents=MAX_DOCUMENTS,
)
print(f"عدد ملفات الشحنات (shards) الناتجة: {len(corpus_files)}")


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

wrote 20,000 documents from wikimedia/wikipedia/20231101.ar into 4 shard files under /content/corpus/wikipedia_ar
عدد ملفات الشحنات (shards) الناتجة: 4


**اختياري:** لإضافة معرفة نوفا الحالية الحقيقية (Supabase) كبيانات تدريب إضافية، أضف Colab Secrets:
`SUPABASE_URL` و `SUPABASE_SERVICE_ROLE_KEY` (نفس طريقة `GITHUB_TOKEN` أعلاه)، ثم شغّل الخلية التالية. إن لم تُضفها، تجاوزها بأمان.

In [6]:
try:
    from data_acquisition import export_nova_knowledge_to_corpus
    own_corpus_path = "/content/corpus/nova_own_knowledge.txt"
    n = export_nova_knowledge_to_corpus(own_corpus_path)
    if n > 0:
        corpus_files.append(own_corpus_path)
    print(f"تمت إضافة {n} فقرة حقيقية من معرفة نوفا الحالية إلى بيانات التدريب.")
except Exception as exc:
    print(f"تم تجاوز هذا المصدر الاختياري (طبيعي إن لم تُضف Colab Secrets الخاصة به): {exc}")


تم تجاوز هذا المصدر الاختياري (طبيعي إن لم تُضف Colab Secrets الخاصة به): Secret 'SUPABASE_URL' not found in Kaggle Secrets, Colab Secrets, or the environment.


### 5) أداة تقسيم النص (Tokenizer)

محفوظة في Google Drive نفسه — إعادة استخدامها تلقائياً في كل جلسة تالية أمر **ضروري وليس اختيارياً**: تدريب أداة جديدة كل مرة يُفسد ربط الأرقام بالكلمات في أي نقطة حفظ سابقة بصمت دون أي خطأ ظاهر.

In [7]:
from pathlib import Path
from text_tokenizer import train_text_tokenizer, ShamTextTokenizer
from model import TEXT_VOCAB_SIZE

tokenizer_path = Path(DRIVE_ROOT) / "sham_small_tokenizer.json"
if tokenizer_path.exists():
    tokenizer = ShamTextTokenizer.load(tokenizer_path)
    print(f"تم إعادة استخدام أداة تقسيم النص المحفوظة في Drive (vocab={tokenizer.vocab_size}) — "
          f"لضمان توافقها مع أي نقطة حفظ سابقة، بدل تدريب أداة جديدة قد تفسد الأرقام المحفوظة.")
else:
    assert corpus_files, "لا توجد ملفات نصية حقيقية — تحقق من نجاح خلية جمع البيانات أعلاه."
    tokenizer = train_text_tokenizer(corpus_files, vocab_size=TEXT_VOCAB_SIZE)
    tokenizer.save(str(tokenizer_path))
    print(f"تم تدريب أداة تقسيم نص حقيقية جديدة (vocab={tokenizer.vocab_size}) وحفظها بشكل دائم في Drive — أول تشغيل فعلي.")


تم إعادة استخدام أداة تقسيم النص المحفوظة في Drive (vocab=32000) — لضمان توافقها مع أي نقطة حفظ سابقة، بدل تدريب أداة جديدة قد تفسد الأرقام المحفوظة.


### 6) بناء بيانات التدريب الفعلية (نوافذ نصية بطول ثابت)

In [ ]:
import torch
from dataset import TextSequenceDataset

SEQ_LEN = 1024

text_dataset = TextSequenceDataset(corpus_files, tokenizer, seq_len=SEQ_LEN)
print(f"عدد نوافذ التدريب الحقيقية الجاهزة: {len(text_dataset):,} (كل نافذة = {SEQ_LEN} رمزاً)")
assert len(text_dataset) >= 32, (
    "عدد نوافذ التدريب قليل جداً — كبّر MAX_DOCUMENTS في خلية جمع البيانات أعلاه وأعد التشغيل من هناك."
)


### 7) إعداد حجم النموذج

نفس منطق دفتر Kaggle — يبدأ بحجم "بداية" آمن يتناسب مع GPU مجاني (T4، ~16GB).

In [ ]:
from model import ShamSmallConfig, ShamSmall, TOTAL_VOCAB_SIZE

MODEL_SIZE = "starter"  # غيّرها إلى "full" إذا كان لديك GPU أقوى

if MODEL_SIZE == "starter":
    model_cfg = ShamSmallConfig(
        vocab_size=TOTAL_VOCAB_SIZE, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4,
        mlp_hidden=2048, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True,
    )
else:
    model_cfg = ShamSmallConfig(vocab_size=TOTAL_VOCAB_SIZE, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True)

model = ShamSmall(model_cfg)
n_params = model.count_parameters()
print(f"تم بناء النموذج: {n_params:,} معامل حقيقي (حجم: {MODEL_SIZE}).")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"سيُستخدم للتدريب: {device}" + ("  (تحذير: لا يوجد GPU — تأكد من Runtime → Change runtime type → GPU)" if device == "cpu" else ""))


### 8) استئناف تدريب سابق إن وُجد

لأن نقاط الحفظ محفوظة في Google Drive نفسه (وليس Output جلسة Kaggle مؤقتة)، هذه الخلية تجدها تلقائياً في أي جلسة تالية بلا أي إجراء إضافي منك — لا حاجة لإضافة أي Input يدوياً كما في Kaggle.

In [ ]:
from pathlib import Path
from checkpoint import load_checkpoint

start_step = 0
resume_optimizer = None

previous_checkpoints = sorted(
    Path(DRIVE_ROOT).rglob("step_*.pt"),
    key=lambda p: int(p.stem.split("_")[1]),
)
if previous_checkpoints:
    last_ckpt = previous_checkpoints[-1]
    model, start_step, _ = load_checkpoint(last_ckpt, map_location=device)
    from train import build_optimizer
    resume_optimizer = build_optimizer(model, lr=3e-4, weight_decay=0.1)
    load_checkpoint(last_ckpt, map_location=device, load_optimizer_into=resume_optimizer)
    print(f"تم استئناف التدريب من نقطة حفظ حقيقية سابقة: {last_ckpt} (الخطوة {start_step:,})")
else:
    print("لم يتم العثور على نقطة حفظ سابقة — سيبدأ التدريب من الصفر (هذا طبيعي في أول تشغيل).")


### 9) قياس السرعة الحقيقية قبل تحديد عدد الخطوات

In [ ]:
import time
from train import TrainConfig, build_optimizer, build_lr_scheduler

CALIBRATION_STEPS = 20

calib_batches = [
    torch.stack([text_dataset[i] for i in range(b, b + 2)])
    for b in range(0, min(len(text_dataset) - 2, CALIBRATION_STEPS * 2 * 4), 2)
][: CALIBRATION_STEPS * 4]

assert calib_batches, "لا توجد بيانات كافية للقياس — كبّر MAX_DOCUMENTS في خلية جمع البيانات."

model.to(device)
model.train()
_calib_optimizer = resume_optimizer or build_optimizer(model, lr=3e-4, weight_decay=0.1)

t0 = time.time()
steps_done = 0
for batch in calib_batches:
    batch = batch.to(device)
    _, loss = model(batch, labels=batch)
    loss.backward()
    _calib_optimizer.step()
    _calib_optimizer.zero_grad()
    steps_done += 1
    if steps_done >= CALIBRATION_STEPS:
        break
elapsed = time.time() - t0
steps_per_second = steps_done / elapsed

# قيمة آمنة تحت حد طول جلسة Colab المجانية الواحدة (حتى ~12 ساعة، وقد
# تُغلق الجلسة أبكر أحياناً حسب سياسة جوجل الديناميكية غير المعلنة) —
# نظام الاستئناف أعلاه يجعل أي إغلاق مبكر آمناً تماماً، فقط أعد تشغيل
# الدفتر وسيكمل من نفسه من آخر نقطة حفظ في Drive.
MAX_TRAINING_HOURS = 10.5
realistic_steps_for_session = int(steps_per_second * MAX_TRAINING_HOURS * 3600 * 0.85)  # هامش أمان 15% إضافي

print(f"سرعة حقيقية مقاسة الآن: {steps_per_second:.3f} خطوة/ثانية على {device}")
print(f"عدد خطوات واقعي يمكن إنجازه ضمن {MAX_TRAINING_HOURS} ساعة (بهامش أمان): {realistic_steps_for_session:,} خطوة")


### 10) التدريب الحقيقي

`max_wall_clock_seconds` يوقف التدريب بنفسه ويحفظ نقطة حفظ حقيقية في Drive قبل الاقتراب من حد الجلسة، بدل انتظار Colab ليقطع الاتصال بلا أي حفظ.

In [ ]:
TOTAL_STEPS = max(realistic_steps_for_session, 200)
num_windows = len(text_dataset) - (len(text_dataset) % 4)
epochs_needed = -(-TOTAL_STEPS // (num_windows // 4))  # للطباعة فقط، تقريب لأعلى
print(f"عدد نوافذ التدريب الحقيقية المتاحة: {num_windows:,} — ستُستخدم عبر {epochs_needed} دورة/دورات (epochs) لإنجاز {TOTAL_STEPS:,} خطوة.")

def _batch_iterator():
    # تبني كل دفعة عند الحاجة فقط بدل تجهيز كل دفعات التدريب مسبقاً في
    # قائمة واحدة ضخمة (وتكرارها) — التجهيز المسبق كان يُبقي نسخة كاملة
    # إضافية من كل نوافذ التدريب في RAM طوال الجلسة، فوق النسخة التي
    # يحتفظ بها text_dataset نفسه أصلاً، وهو ما أسقط جلسة Colab فعلياً
    # بعد استنفاد كل RAM المتاح (حادثة حقيقية، 2026-09-15).
    while True:
        for b in range(0, num_windows, 4):
            yield torch.stack([text_dataset[i] for i in range(b, b + 4)])

import itertools
batches = itertools.islice(_batch_iterator(), TOTAL_STEPS)

train_cfg = TrainConfig(
    seq_len=SEQ_LEN,
    batch_size=4,
    grad_accum_steps=4,          # حجم دفعة فعلي = 16
    lr=3e-4,
    warmup_steps=max(50, TOTAL_STEPS // 100),
    total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir=f"{DRIVE_ROOT}/checkpoints",
    checkpoint_every=200,
    log_every=20,
    max_wall_clock_seconds=MAX_TRAINING_HOURS * 3600,
)

from train import train
loss_history = train(
    model, batches, train_cfg, device=device,
    start_step=start_step, resume_optimizer=resume_optimizer or _calib_optimizer,
)

print(f"\nانتهى التدريب على {len(loss_history):,} خطوة حقيقية.")
print(f"متوسط الخسارة (loss) في أول 10 خطوات: {sum(loss_history[:10]) / min(10, len(loss_history)):.4f}")
print(f"متوسط الخسارة (loss) في آخر 10 خطوات: {sum(loss_history[-10:]) / min(10, len(loss_history)):.4f}")


### 11) حفظ النتيجة النهائية

كل نقاط الحفظ محفوظة بالفعل بشكل دائم في Google Drive أثناء التدريب نفسه — لا حاجة لأي خطوة تصدير إضافية كما في Kaggle. هذه الخلية فقط تحفظ نقطة نهائية صريحة.

In [ ]:
from checkpoint import save_checkpoint

final_step = start_step + len(loss_history)
save_checkpoint(f"{DRIVE_ROOT}/checkpoints/final.pt", model, final_step)
print(f"تم حفظ النقطة النهائية عند الخطوة {final_step:,} في {DRIVE_ROOT}/checkpoints/final.pt")
print(f"أداة تقسيم النص محفوظة في {DRIVE_ROOT}/sham_small_tokenizer.json")
print("\nكل شيء محفوظ بشكل دائم في Drive بالفعل — فقط أعد فتح هذا الدفتر لاحقاً واضغط Run all ليكمل من هنا تلقائياً.")
